# 05a — OpenSim 運動學基礎：Frames、Coordinates 與 Marker-based IK
### *Upper-limb kinematics fundamentals on the bundled `arm26` model*

本 notebook 是本章「**先通用基礎**」的部分，先把 OpenSim 運動學的可遷移概念打通，
再到 [05b](05b_kinatrax_angle_import_and_mapping.ipynb) / [05c](05c_scapula_kinematics_generation.ipynb) 接你的
Kinatrax→UCL 研究。這裡刻意用 **marker-based Scale + Inverse Kinematics (IK)** —
即使你的真實資料是關節角度 (見 05b)，marker-IK 仍是資格考核心、也是你 ≤120° 抬舉
體表 marker validation trial 會用到的路徑。

**學習目標**
1. 理解 OpenSim 的 `Frame` / `Joint` / `Coordinate`，以及一個 coordinate 值 $q$ 如何
   對應到 parent↔child frame 之間的 spatial transform。
2. 理解 marker model，並用 forward-driven `arm26` **合成 (synthetic)** 一份 marker `.trc`。
3. 把 IK 寫成加權最小平方 (weighted least squares) 問題並實際求解，驗證回推角度。

> ⚙️ **環境需求**：本 notebook 需要 conda 環境內的 OpenSim 4.x Python (`import opensim`)
> 以及內建的 `arm26.osim`。純資料處理與繪圖不需 OpenSim，但模型相關 cell 需要。


In [ ]:
# --- 讓 notebook 找得到本章的 src/ 模組 (put chapter src/ on sys.path) ---
import sys, pathlib
CHAPTER = pathlib.Path.cwd()
if CHAPTER.name == "notebooks":
    CHAPTER = CHAPTER.parent          # 允許從 notebooks/ 內啟動
sys.path.insert(0, str(CHAPTER / "src"))

import numpy as np
import matplotlib.pyplot as plt

import osim_kinematics_io as kio
print('src loaded:', kio.__file__)


## 0. 環境檢查 (environment check)


In [ ]:
import opensim as osim
print("OpenSim version:", kio.check_opensim())


In [ ]:
# --- 定位 arm26.osim (OpenSim 內建範例模型 / bundled example model) ---
# arm26 隨 OpenSim GUI 安裝，或見 github.com/opensim-org/opensim-models。
# 設環境變數 ARM26 指向 arm26.osim，或把它放進本章 data/。
import os, pathlib

def find_arm26():
    cands = []
    if os.environ.get("ARM26"):
        cands.append(pathlib.Path(os.environ["ARM26"]))
    cands.append(CHAPTER / "data" / "arm26.osim")
    # 常見安裝路徑 (Windows / conda opensim-org / macOS)
    for root in [r"C:/OpenSim 4.5/Models/Arm26",
                 r"C:/OpenSim 4.4/Models/Arm26",
                 str(pathlib.Path.home() / "opensim-models" / "Pipelines" / "Arm26"),
                 "/Applications/OpenSim 4.5/OpenSim.app/Contents/Resources/Models/Arm26"]:
        cands.append(pathlib.Path(root) / "arm26.osim")
    for c in cands:
        if c.is_file():
            return str(c)
    raise FileNotFoundError(
        "找不到 arm26.osim。請設環境變數 ARM26 指向該檔，或複製到 "
        f"{CHAPTER / 'data'}。arm26 隨 OpenSim 安裝，亦可自 "
        "github.com/opensim-org/opensim-models 取得。")

ARM26 = find_arm26()
print('arm26:', ARM26)


## 1. Frames、Joints、Coordinates：一個 $q$ 如何變成一個 transform

OpenSim 的一個 `Joint` 連接 **parent frame** 與 **child frame**（兩者常是
`PhysicalOffsetFrame`，即固定在某 body 上、帶常數位移/旋轉的座標系）。`Joint` 的
`SpatialTransform` 把每個 `Coordinate` 值 $q_i$ 透過一條 transform axis
$(\hat{a}_i,\; f_i(q_i))$ 疊加成 parent→child 的 6-DOF spatial transform：

$$ {}^{P}X_{C}(q) \;=\; \prod_i \exp\!\big(\, [\hat a_i]\, f_i(q_i)\,\big). $$

對 `arm26`，`r_shoulder_elev` 與 `r_elbow_flex` 都是 1-DOF 旋轉座標。下面列出模型
座標並檢視關節的 parent/child frame 與其在 ground 中的位置。


In [ ]:
model = kio.load_model(ARM26)
coords = kio.list_coordinates(model)
for c in coords:
    print(f"{c['name']:16s} joint={c['joint']:14s} type={c['motion_type']:11s} "
          f"default={np.rad2deg(c['default_value']) if c['is_rotational'] else c['default_value']:7.2f}"
          f"{'deg' if c['is_rotational'] else ''}  "
          f"range=[{c['range_min']:.2f},{c['range_max']:.2f}]")
    print("   state:", c['value_state'])


In [ ]:
# 檢視 elbow joint 的 parent/child frame，並看 coordinate 值 -> ground transform
state = model.initSystem()

def report_frames(joint_name):
    jnt = model.getJointSet().get(joint_name)
    pf, cf = jnt.getParentFrame(), jnt.getChildFrame()
    for tag, fr in (("parent", pf), ("child", cf)):
        T = fr.getTransformInGround(state)
        p = T.p()
        print(f"  {tag:6s} frame '{fr.getName()}'  origin_in_ground="
              f"({p.get(0):+.3f},{p.get(1):+.3f},{p.get(2):+.3f})")

for j in ("r_shoulder", "r_elbow"):
    try:
        print("joint:", j); report_frames(j)
    except Exception as e:
        print("joint", j, "->", e, "(名稱依模型而定，可用 getJointSet 列出)")

# 掃一個 coordinate 值，觀察 child frame 在 ground 的位置隨 q 改變
elbow = model.getCoordinateSet().get("r_elbow_flex")
child = model.getJointSet().get("r_elbow").getChildFrame()
for deg in (0, 45, 90):
    elbow.setValue(state, np.deg2rad(deg)); model.realizePosition(state)
    p = child.getTransformInGround(state).p()
    print(f"  r_elbow_flex={deg:3d}deg -> child origin ({p.get(0):+.3f},{p.get(1):+.3f},{p.get(2):+.3f})")


## 2. Marker model 與合成 `.trc`

真實 marker-IK 需要一份實驗 marker 軌跡 (`.trc`)。這裡沒有實驗資料，所以我們
**用模型自己生成 (synthetic)**：先在 body 上掛幾顆 marker，prescribe 一段平滑的
肩/肘運動，逐格記錄 marker 在 ground 的位置，寫成 `.trc`。之後 IK 應該能把角度
回推出來（因為 ground truth 已知）。

> 註：`arm26` 預設不含 marker set，因此下面**若模型沒有 marker 就程式化加上**。
> 真實研究裡 marker 名稱/位置來自你的 marker protocol。


In [ ]:
# (a) 確保模型有 marker：若沒有就掛幾顆在 humerus / forearm 上
model = kio.load_model(ARM26)
ms = model.getMarkerSet()

def body_names(m):
    bs = m.getBodySet(); return [bs.get(i).getName() for i in range(bs.getSize())]
print("bodies:", body_names(model))

if ms.getSize() == 0:
    bs = model.getBodySet()
    hum = bs.get("r_humerus")
    # forearm body 名稱在不同版本可能是 r_ulna_radius_hand
    fore = None
    for cand in ("r_ulna_radius_hand", "r_ulna", "r_radius"):
        try:
            fore = bs.get(cand); break
        except Exception:
            continue
    to_add = [("HUM1", hum, osim.Vec3(0.00, -0.15, 0.03)),
              ("HUM2", hum, osim.Vec3(0.03, -0.25, 0.00)),
              ("FOR1", fore, osim.Vec3(0.00, -0.12, 0.02)),
              ("FOR2", fore, osim.Vec3(0.02, -0.22, 0.00))]
    for nm, body, loc in to_add:
        mk = osim.Marker(nm, body, loc); model.addMarker(mk)
    print("added markers:", [ms.get(i).getName() for i in range(model.getMarkerSet().getSize())])
model.finalizeConnections()


In [ ]:
# (b) prescribe 平滑肩/肘運動，逐格取 marker 的 ground 座標
fs = 100.0
t = np.arange(0.0, 1.0, 1.0 / fs)
bump = (1.0 - np.cos(2 * np.pi * 0.5 * t)) / 2.0            # 0->1->~0 平滑
truth_deg = {"r_shoulder_elev": 10 + 70 * bump,            # deg
             "r_elbow_flex":    5 + 100 * bump}

state = model.initSystem()
mset = model.getMarkerSet()
mk_names = [mset.get(i).getName() for i in range(mset.getSize())]
traj = {nm: np.zeros((t.size, 3)) for nm in mk_names}

for k in range(t.size):
    for cn, series in truth_deg.items():
        model.getCoordinateSet().get(cn).setValue(state, np.deg2rad(series[k]))
    model.realizePosition(state)
    for i, nm in enumerate(mk_names):
        g = mset.get(nm).getLocationInGround(state)
        traj[nm][k] = [g.get(0), g.get(1), g.get(2)]
print("marker frames:", t.size, "markers:", mk_names)


In [ ]:
# (c) 寫成 .trc (單位 m)。使用 TimeSeriesTableVec3 + TRCFileAdapter。
trc_path = str(CHAPTER / "data" / "arm26_synthetic.trc")
table = osim.TimeSeriesTableVec3()
table.setColumnLabels(mk_names)
for k in range(t.size):
    row = osim.RowVectorVec3([osim.Vec3(*traj[nm][k]) for nm in mk_names])
    table.appendRow(float(t[k]), row)
table.addTableMetaDataString("DataRate", str(fs))
table.addTableMetaDataString("CameraRate", str(fs))
table.addTableMetaDataString("Units", "m")
osim.TRCFileAdapter().write(table, trc_path)
print("wrote", trc_path)


## 3. IK = 加權最小平方 (weighted least squares)

在每個時間點，IK 解

$$ \min_{q}\; \sum_{i\in\text{markers}} w_i\,\big\lVert x_i^{\exp} - x_i(q)\big\rVert^2
   \;+\; \sum_{j\in\text{coords}} \omega_j\,\big(q_j^{\exp}-q_j\big)^2, $$

其中 $x_i(q)$ 是 marker $i$ 由模型正向運動學算出的位置、$w_i$ 是 marker weight。
權重越大代表越信任該 marker（或該 coordinate 的先驗）。下面用 `InverseKinematicsTool`
＋ `IKTaskSet` 設定 marker weight 並求解。


In [ ]:
ik = osim.InverseKinematicsTool()
ik.setModel(model)
ik.setMarkerDataFileName(trc_path)
ik.setStartTime(float(t[0]))
ik.setEndTime(float(t[-1]))

# 每顆 marker 一個 IKMarkerTask，weight=1
tasks = osim.IKTaskSet()
for nm in mk_names:
    task = osim.IKMarkerTask()
    task.setName(nm); task.setWeight(1.0); task.setApply(True)
    tasks.cloneAndAppend(task)
ik.set_IKTaskSet(tasks)

ik_mot = str(CHAPTER / "data" / "arm26_ik.mot")
ik.setOutputMotionFileName(ik_mot)
ik.run()
print("IK done ->", ik_mot)


In [ ]:
# 讀回 IK 結果，與 ground truth 比對 (IK .mot 依模型 inDegrees 旗標，此處為 deg)
times, cols = kio.read_mot(ik_mot)
print("IK columns:", list(cols)[:6], "...  inDegrees:", kio.mot_is_in_degrees(ik_mot))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, cn in zip(ax, ("r_shoulder_elev", "r_elbow_flex")):
    a.plot(t, truth_deg[cn], "k-", lw=3, alpha=.4, label="ground truth")
    if cn in cols:
        a.plot(times, cols[cn], "r--", label="IK recovered")
        rms = np.sqrt(np.mean((np.interp(t, times, cols[cn]) - truth_deg[cn])**2))
        a.set_title(f"{cn}  (RMS={rms:.3f} deg)")
    a.set_xlabel("time (s)"); a.set_ylabel("deg"); a.legend()
plt.tight_layout(); plt.show()


## 4. Scaling 小記 (ScaleTool, conceptual)

真實流程在 IK 前會先 **scale**：用一個靜態 T-pose trial 的 marker 距離，對 generic
模型逐段縮放，讓 model marker 與 experimental marker 對齊，並調整 body 慣量。API 是
`osim.ScaleTool(setup.xml)`，內含 `GenericModelMaker` / `ModelScaler` / `MarkerPlacer`
三段。本 notebook 用合成資料、模型即 ground truth，故略過 scaling；你的 validation
trial 需要它。

## 小結
- 一個 coordinate 值透過 `SpatialTransform` 決定 parent↔child 的 6-DOF transform。
- Marker-IK 是逐格的加權最小平方；weight 決定信任度。
- 你的主資料是**角度**而非 marker → 見 [05b](05b_kinatrax_angle_import_and_mapping.ipynb) 的角度驅動路徑與慣例映射。
